# Capítulo 2 — Gráficos de Séries Temporais
**Livro:** Forecasting: Principles and Practice, the Pythonic Way  
**Pesquisa:** Previsão de Demanda com Séries Hierárquicas

---

## O que você vai aprender aqui

O cap. 2 é sobre **enxergar** antes de modelar. Toda análise de forecasting começa com visualização.
Quem pula essa etapa modela no escuro.

Seções do capítulo:
- 2.1 DataFrames e estrutura de séries temporais
- 2.2 Gráficos temporais (time plots)
- 2.3 Padrões: tendência, sazonalidade e ciclo
- 2.4 Gráficos sazonais
- 2.5 Gráficos de sub-séries sazonais
- 2.6 Diagramas de dispersão e correlação
- 2.7 Gráficos de lag
- 2.8 Autocorrelação (ACF)
- 2.9 Ruído branco (White Noise)

---

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.graphics.tsaplots import plot_acf
from scipy.stats import pearsonr
import warnings
warnings.filterwarnings('ignore')

# instale se necessário
# !pip install utilsforecast --quiet
from utilsforecast.plotting import plot_series

plt.style.use('ggplot')
plt.rcParams.update({'figure.figsize': (10, 4), 'figure.dpi': 100})

print('Ambiente pronto.')

---
## 2.1 DataFrames e Estrutura de Séries Temporais

Uma série temporal é uma sequência de observações numéricas com registro de tempo.
O pandas usa dois tipos principais para representar tempo:

- `pd.Timestamp` — instante no tempo (ex: 2020-01-01 08:00)
- `pd.Period` — intervalo de tempo (ex: janeiro de 2020)

Na prática do nixtlaverse, sempre usamos `Timestamp` armazenado na coluna `ds`.

In [ ]:
# diferença entre Timestamp e Period
print('Timestamp:', repr(pd.Timestamp('2020-01')))
print('Period:   ', repr(pd.Period('2020-01')))
print()

# criando sequências de datas
ts_range = pd.date_range('2020-01-01', '2020-01-02', freq='8h')
print('date_range a cada 8h:')
print(ts_range)
print()

# convertendo para período
print('Convertido para período mensal:')
print(ts_range.to_period(freq='M'))

In [ ]:
# o acessor .dt permite manipular datas dentro de uma Series
df = pd.DataFrame({'ts': pd.to_datetime(['2020-01-01', '2020-01-02', '2020-01-03'])})
df = df.assign(
    periodo=df['ts'].dt.to_period('M'),
    ano=df['ts'].dt.year,
    dia_semana=df['ts'].dt.strftime('%A, %B %-d'),
).set_index('ts')
df

# .dt.year, .dt.month, .dt.quarter, .dt.strftime() — use isso sempre ao manipular datas

In [ ]:
# exemplo com dataset de farmácias australianas (PBS)
# mostra como filtrar, agregar e renomear — operações essenciais em qualquer pipeline
pbs = (
    pd.read_csv('data/PBS_unparsed.csv', parse_dates=['Month'])
    [['Month', 'Concession', 'Type', 'ATC1', 'ATC2', 'Scripts', 'Cost']]
)

# filtra classe ATC2 = A10 (medicamentos antidiabéticos)
# agrega por mês e converte custo para milhões
total_cost_df = (
    pbs.loc[pbs['ATC2'] == 'A10']
    .drop(columns=['ATC1', 'ATC2'])
    .groupby('Month', as_index=False)
    .agg({'Cost': 'sum'})
    .assign(Cost=lambda x: (x['Cost'] / 1e6).round(2))
)

print(f'Shape: {total_cost_df.shape}')
print(f'Período: {total_cost_df["Month"].min()} a {total_cost_df["Month"].max()}')
total_cost_df.head()

**Período sazonal — tabela de referência:**

| Frequência dos dados | Período sazonal |
|----------------------|------------------|
| Anual | 1 |
| Trimestral | 4 |
| Mensal | 12 |
| Semanal | 52 |
| Diário (sazonalidade semanal) | 7 |
| Diário (sazonalidade anual) | 365 |
| Por hora | 24 / 168 / 8766 |

Esse valor é o `period` do STL, o `season_length` do SeasonalNaive, e o `m` do ARIMA sazonal.
**Errar o período sazonal quebra o modelo.**

---
## 2.2 Gráficos Temporais (Time Plots)

O primeiro gráfico de qualquer análise é sempre o gráfico temporal — observações no eixo y, tempo no eixo x.
Revela tendência, sazonalidade, outliers e mudanças estruturais.

In [ ]:
# carrega companhia aérea Ansett (Melbourne-Sydney)
melsyd = (
    pd.read_csv('data/ansett.csv', parse_dates=['ds'])
    .loc[lambda x: (x['Airports'] == 'MEL-SYD') & (x['Class'] == 'Economy')]
    .rename(columns={'Airports': 'unique_id'})
    .assign(y=lambda x: x['y'] / 1000)
)

fig, ax = plt.subplots()
ax.plot(melsyd['ds'], melsyd['y'], color='steelblue', linewidth=1)
ax.set_title('Ansett Airlines — Classe Econômica: Melbourne-Sydney')
ax.set_xlabel('Semana')
ax.set_ylabel('Passageiros (mil)')
plt.tight_layout()
plt.show()

# o que o gráfico revela:
# 1989: zero passageiros — greve trabalhista
# 1992: redução — substituição de assentos econômicos por executivos
# início de cada ano: queda — efeito feriados
# qualquer modelo precisa capturar esses padrões

In [ ]:
# gráfico temporal do dataset de medicamentos antidiabéticos
fig, ax = plt.subplots()
ax.plot(total_cost_df['Month'], total_cost_df['Cost'], color='steelblue', linewidth=1.5)
ax.set_title('Vendas mensais de medicamentos antidiabéticos — Austrália')
ax.set_xlabel('Mês')
ax.set_ylabel('$ (milhões)')
plt.tight_layout()
plt.show()

# o que você vê:
# tendência crescente clara ao longo dos anos
# sazonalidade forte que aumenta de intensidade com o tempo (sazonalidade multiplicativa)
# queda abrupta no início de cada ano — pacientes estocam no final do ano

---
## 2.3 Padrões de Séries Temporais

Três padrões fundamentais a identificar em qualquer série:

| Padrão | Definição | Como identificar |
|--------|-----------|------------------|
| **Tendência** | Crescimento ou queda sistemática de longo prazo | Linha geral sobe ou desce |
| **Sazonalidade** | Padrão repetitivo em intervalos fixos e conhecidos | Picos/vales regulares (período fixo) |
| **Ciclo** | Flutuações sem frequência fixa (geralmente > 2 anos) | Ondas irregulares — confundido com tendência |

**Diferença crítica entre sazonalidade e ciclo:**
- Sazonalidade tem período fixo e conhecido (12 meses, 7 dias)
- Ciclo não tem período fixo — é governado por condições econômicas

**Na sua pesquisa:** previsão de demanda de estoque geralmente envolve tendência + sazonalidade.
Ciclos aparecem em dados de demanda de construção civil, commodities e setores cíclicos.

In [ ]:
# visualiza 4 exemplos de padrões diferentes
hsales = pd.read_csv('data/hsales.csv', parse_dates=['ds'])
ustreas = pd.read_csv('data/ustreas.csv', parse_dates=['ds'])
aus_prod = pd.read_csv('data/aus_production.csv', parse_dates=['ds'])
gafa = pd.read_csv('data/gafa_stock.csv', parse_dates=['ds'])
goog = gafa[gafa['unique_id'] == 'GOOG_Close'].copy()
goog_diff = goog.assign(y=goog['y'].diff()).dropna()

fig, axes = plt.subplots(2, 2, figsize=(12, 7))

axes[0,0].plot(hsales['ds'], hsales['y'], color='steelblue', linewidth=0.8)
axes[0,0].set_title('Vendas mensais de imóveis — EUA\n(Tendência + Sazonalidade + Ciclo)')

axes[0,1].plot(ustreas['ds'], ustreas['y'], color='darkorange', linewidth=0.8)
axes[0,1].set_title('Contratos do Tesouro dos EUA\n(Tendência de queda, sem sazonalidade)')

elec = aus_prod[['ds', 'Electricity']].dropna()
axes[1,0].plot(elec['ds'], elec['Electricity'], color='green', linewidth=0.8)
axes[1,0].set_title('Produção trimestral de eletricidade — Austrália\n(Tendência forte + Sazonalidade)')

axes[1,1].plot(goog_diff['ds'], goog_diff['y'], color='gray', linewidth=0.5)
axes[1,1].set_title('Variação diária do preço do Google\n(Sem padrão — ruído branco)')

plt.tight_layout()
plt.show()

---
## 2.4 Gráficos Sazonais

O gráfico sazonal sobrepõe os dados de cada ciclo (ano, semana, dia) no mesmo eixo x.
Revela o padrão sazonal com mais clareza e mostra se ele muda ao longo do tempo.

In [ ]:
# gráfico sazonal dos medicamentos antidiabéticos
df = total_cost_df.assign(
    Ano=total_cost_df['Month'].dt.year,
    Mes_num=total_cost_df['Month'].dt.month,
)

anos_unicos = df['Ano'].unique()
palette = sns.color_palette('husl', n_colors=len(anos_unicos))

fig, ax = plt.subplots(figsize=(11, 5))
sns.lineplot(data=df, x='Mes_num', y='Cost',
             hue='Ano', palette=palette, legend=False, ax=ax)

ax.set(
    title='Gráfico Sazonal — Vendas de Medicamentos Antidiabéticos',
    xlabel='Mês', ylabel='$ (milhões)',
    xticks=range(1, 13),
    xticklabels=['Jan','Fev','Mar','Abr','Mai','Jun',
                 'Jul','Ago','Set','Out','Nov','Dez']
)

# rotula o ano no final de cada linha
min_ano = anos_unicos.min()
for ano, subset in df.groupby('Ano'):
    x = subset['Mes_num'].iloc[-1] + 0.1
    y = subset['Cost'].iloc[-1]
    color = palette[ano - min_ano]
    ax.text(x, y, str(ano), ha='left', va='center', fontsize=8, color=color)

plt.tight_layout()
plt.show()

# o que você vê:
# pico em janeiro todo ano — estoque no fim do ano civil
# padrão consistente ao longo dos anos (sazonalidade estável)
# vendas crescem a cada ano — tendência crescente

In [ ]:
# sazonalidade múltipla — demanda de eletricidade em Victoria (30 minutos)
# mostra padrão diário: pico de manhã e tarde/noite
vic_elec_df = pd.read_csv('data/vic_elec.csv', parse_dates=['ds'])
vic_elec_demand = vic_elec_df[vic_elec_df['unique_id'] == 'Demand'].copy()

df_hora = vic_elec_demand.assign(
    hora_min=lambda x: x['ds'].dt.strftime('%H:%M'),
    dia=lambda x: x['ds'].dt.date,
)

fig, ax = plt.subplots(figsize=(11, 4))
sns.lineplot(data=df_hora, x='hora_min', y='y',
             hue='dia', palette='husl', legend=False, ax=ax)

# mostra só algumas marcações no eixo x
ticks_unicos = df_hora['hora_min'].unique()
ax.set_xticks(range(0, len(ticks_unicos), 4))
ax.set_xticklabels(ticks_unicos[::4], rotation=45)

ax.set(
    title='Padrão Diário — Demanda de Eletricidade em Victoria\n(cada linha = um dia)',
    xlabel='Hora', ylabel='MWh'
)
plt.tight_layout()
plt.show()

# aplicação na sua pesquisa:
# demanda de combustível em frotas tem sazonalidade múltipla:
# padrão diário (mais consumo durante o dia), semanal e sazonal anual

---
## 2.5 Gráficos de Sub-séries Sazonais

Agrupa os dados por estação em mini-gráficos separados.
A linha horizontal azul mostra a média de cada estação.
Revela mudanças dentro de cada estação ao longo do tempo.

In [ ]:
# sub-séries dos medicamentos antidiabéticos
df = total_cost_df.assign(
    ano=total_cost_df['Month'].dt.year,
    mes_nome=total_cost_df['Month'].dt.month_name(),
    mes_idx=total_cost_df['Month'].dt.month,
)

fig, axes = plt.subplots(1, 12, figsize=(16, 4), sharey=True)

for ax, ((_, mes_nome), mes_df) in zip(axes, df.groupby(['mes_idx', 'mes_nome'])):
    media = mes_df['Cost'].mean()
    ax.plot(mes_df['ano'], mes_df['Cost'], color='k', linewidth=1)
    ax.axhline(media, color='steelblue', linewidth=1.5)
    ax.set(title=mes_nome[:3], xlabel='')
    ax.tick_params(axis='x', rotation=90)

fig.suptitle('Sub-séries Sazonais — Medicamentos Antidiabéticos')
fig.supxlabel('Ano')
fig.supylabel('$ (milhões)')
plt.tight_layout()
plt.show()

# a linha azul = média histórica de cada mês
# se a linha preta (valores reais) está sempre acima da azul nos últimos anos = tendência crescente

In [ ]:
# sub-séries do turismo australiano por estado e trimestre
# esse dataset tem estrutura hierárquica: importantíssimo pro cap. 11
tourism = pd.read_csv('data/tourism.csv', parse_dates=['ds'])

# agrega viagens de férias por estado e trimestre
trips = (
    tourism.loc[lambda x: x['Purpose'] == 'Holiday']
    .groupby(['State', 'ds'], as_index=False)
    .agg({'y': 'sum'})
    .assign(
        Trimestre='Q' + tourism.loc[lambda x: x['Purpose'] == 'Holiday']
            .groupby(['State', 'ds'])['ds'].first().reset_index()['ds'].dt.quarter.astype('string'),
        Ano=lambda x: x['ds'].dt.year,
    )
)

print('Estados:', trips['State'].unique())
print('Shape:', trips.shape)
trips.head()

---
## 2.6 Diagramas de Dispersão e Correlação

Úteis para explorar relações **entre** séries temporais — especialmente quando há variáveis externas (covariáveis/regressores).

**Correlação de Pearson (r):**
- Varia entre -1 e +1
- r = +1: relação linear perfeita positiva
- r = -1: relação linear perfeita negativa
- r = 0: sem relação linear

**Atenção:** correlação mede só relação **linear**. Relações não lineares podem ser fortes mas ter r baixo.

In [ ]:
# scatter plot: demanda de eletricidade vs temperatura
# mostra relação não linear — calor e frio aumentam demanda
elec_2014 = (
    vic_elec_df.loc[lambda x: x['ds'].dt.year == 2014]
    .pivot(index='ds', columns='unique_id', values='y')
    .reset_index()
)

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(elec_2014['Temperature'], elec_2014['Demand'],
           alpha=0.3, s=5, color='steelblue')
ax.set(
    title='Demanda de Eletricidade vs Temperatura — Victoria 2014',
    xlabel='Temperatura (graus Celsius)',
    ylabel='Demanda (GW)'
)

# calcula correlação
r, p = pearsonr(elec_2014['Temperature'].dropna(), elec_2014['Demand'].dropna())
ax.text(0.05, 0.95, f'r = {r:.2f}', transform=ax.transAxes,
        fontsize=12, va='top')

plt.tight_layout()
plt.show()

# a correlação r=0.28 parece baixa, mas o gráfico mostra relação não linear forte
# isso é por que SEMPRE olhamos o gráfico, não só o número

In [ ]:
# matriz de dispersão — turismo por estado
# mostra correlação entre todas as séries de uma vez
# útil pra entender estrutura hierárquica dos dados
visitors = tourism.groupby(['State', 'ds'], as_index=False)['y'].sum()
df_pivot = visitors.pivot(index='ds', columns='State', values='y')

def corrfunc(x, y, **kws):
    r, pvalue = pearsonr(x.dropna(), y.dropna())
    ax = plt.gca()
    ax.annotate(
        f'r={r:.2f}{"*" if pvalue < 0.05 else ""}',
        xy=(0.5, 0.5), xycoords='axes fraction',
        ha='center', va='center', fontsize=9
    )

g = sns.PairGrid(df_pivot, height=1.2)
g.map_lower(sns.scatterplot, s=3, alpha=0.5)
g.map_upper(corrfunc)
g.map_diag(sns.kdeplot, lw=1.5)
g.figure.suptitle('Matriz de Correlação — Turismo por Estado', y=1.01)
plt.tight_layout()
plt.show()

# estados vizinhos (NSW, VIC, SA) têm correlações mais altas
# isso é esperado — padrões sazonais similares em regiões próximas
# na sua pesquisa: lojas próximas tendem a ter demanda correlacionada

---
## 2.7 Gráficos de Lag

O gráfico de lag plota $y_t$ contra $y_{t-k}$ para diferentes valores de k.
Revela se o valor atual se correlaciona com valores passados — e com qual defasagem.

**Na previsão de demanda:**
- Lag 7 forte = demanda de hoje depende da de 7 dias atrás (padrão semanal)
- Lag 12 forte = demanda de hoje depende do mesmo mês do ano passado (padrão anual)
- Isso define quais lags incluir como features em modelos de ML

In [ ]:
# gráfico de lag — produção trimestral de cerveja australiana
beer = (
    pd.read_csv('data/aus_production.csv', parse_dates=['ds'])
    [['ds', 'Beer']]
    .loc[lambda x: x['ds'] >= '2000']
    .assign(
        Trimestre=lambda x: 'Q' + x['ds'].dt.quarter.astype('string'),
        y=lambda x: x['Beer']
    )
)

cores_trimestre = {'Q1': 'steelblue', 'Q2': 'darkorange', 'Q3': 'green', 'Q4': 'red'}

fig, axes = plt.subplots(3, 3, figsize=(9, 8), sharex=True, sharey=True)

for i, ax in enumerate(axes.flat):
    lag = i + 1
    df_lag = beer.assign(lag_y=beer['y'].shift(lag)).dropna()
    for trim, grp in df_lag.groupby('Trimestre'):
        ax.scatter(grp['lag_y'], grp['y'],
                   color=cores_trimestre[trim], label=trim, s=20, alpha=0.8)
    ax.set_title(f'lag {lag}', fontsize=9)
    ax.set_aspect('equal')

axes[1, -1].legend(loc='center left', bbox_to_anchor=(1.05, 0.5),
                   title='Trimestre', frameon=False)
fig.supxlabel('lag(Beer, k)')
fig.supylabel('Beer')
fig.suptitle('Gráficos de Lag — Produção de Cerveja (Trimestral)')
plt.tight_layout()
plt.show()

# lag 4 e 8: correlação positiva forte — sazonalidade anual
# lag 2 e 6: correlação negativa — picos vs vales
# isso confirma que period=4 é o correto para séries trimestrais

---
## 2.8 Autocorrelação (ACF)

A autocorrelação mede a correlação de uma série com versões defasadas dela mesma.

Fórmula do coeficiente de autocorrelação no lag k:

$$r_k = \frac{\sum_{t=k+1}^{T}(y_t - \bar{y})(y_{t-k} - \bar{y})}{\sum_{t=1}^{T}(y_t - \bar{y})^2}$$

**Como ler o gráfico ACF (correlograma):**
- Barras fora da banda cinza = autocorrelação significativa
- Barras altas nos lags 4, 8, 12... = sazonalidade trimestral
- Barras que decrescem lentamente = tendência na série
- Forma de escada com ondas = tendência + sazonalidade juntas

In [ ]:
# ACF da produção de cerveja (trimestral)
fig, ax = plt.subplots(figsize=(10, 4))
plot_acf(beer['y'], lags=16, ax=ax,
         zero=False, bartlett_confint=False, auto_ylims=True)
ax.set(
    title='ACF — Produção Trimestral de Cerveja',
    xlabel='Lag (trimestres)', ylabel='Autocorrelação'
)
plt.tight_layout()
plt.show()

# r4 e r8 são os maiores — confirma sazonalidade trimestral
# r2 e r6 são negativos — picos vs vales (lag = metade do período)

In [ ]:
# ACF dos medicamentos antidiabéticos (mensal — tendência + sazonalidade)
fig, ax = plt.subplots(figsize=(10, 4))
plot_acf(total_cost_df['Cost'], lags=48, ax=ax,
         zero=False, bartlett_confint=False, auto_ylims=True)
ax.set(
    title='ACF — Medicamentos Antidiabéticos (Mensal)',
    xlabel='Lag (meses)', ylabel='Autocorrelação'
)
plt.tight_layout()
plt.show()

# queda lenta = tendência
# forma de 'vieira' (scalloped) = sazonalidade
# combinação dos dois = tendência + sazonalidade

In [ ]:
# calcula os valores da ACF manualmente
acf_values = sm.tsa.acf(beer['y'], nlags=9, fft=False, bartlett_confint=False)
acf_df = pd.Series(acf_values, name='ACF').to_frame().rename_axis('lag')
print('Valores da ACF:')
print(acf_df[1:].round(3))
print()
print('Lag 4 e 8 são os maiores — confirmam sazonalidade trimestral')

---
## 2.9 Ruído Branco (White Noise)

Uma série é **ruído branco** quando não apresenta autocorrelação — os valores não dependem dos anteriores.

**Por que isso importa na sua pesquisa:**
- Os **resíduos** do modelo precisam ser ruído branco
- Se os resíduos têm autocorrelação, o modelo não capturou toda a informação
- Testar se os resíduos são ruído branco é o diagnóstico padrão de qualquer modelo de forecasting

**Regra:** em uma série de comprimento T, esperamos que 95% das barras da ACF fiquem dentro de $\pm 1.96/\sqrt{T}$

In [ ]:
# gera uma série de ruído branco
rng = np.random.RandomState(30)
wn = pd.DataFrame({
    'y': rng.normal(0, 1, 50),
    'ds': np.arange(1, 51),
    'unique_id': 'wn'
})

fig, axes = plt.subplots(2, 1, figsize=(10, 6))

# série
axes[0].plot(wn['ds'], wn['y'], color='steelblue', linewidth=1)
axes[0].axhline(0, color='gray', linestyle='--', alpha=0.5)
axes[0].set_title('Série de Ruído Branco')
axes[0].set_xlabel('Observação')
axes[0].set_ylabel('Valor')

# ACF
plot_acf(wn['y'], lags=16, ax=axes[1],
         zero=False, bartlett_confint=False, auto_ylims=True)
axes[1].set_title('ACF — Ruído Branco (quase tudo dentro da banda)')
axes[1].set_xlabel('Lag')
axes[1].set_ylabel('Autocorrelação')

# limite teórico para T=50
limite = 1.96 / np.sqrt(50)
print(f'Limite teórico (±1.96/√50): ±{limite:.3f}')
print('Se quase todas as barras estão dentro desse limite, a série é ruído branco')

plt.tight_layout()
plt.show()

---
## Consolidação — O que você precisa saber do cap. 2

| Ferramenta | Quando usar | O que revela |
|------------|-------------|---------------|
| Time plot | Sempre primeiro | Tendência, outliers, mudanças estruturais |
| Gráfico sazonal | Quando suspeitar de sazonalidade | Padrão sazonal e sua estabilidade |
| Sub-séries sazonais | Quando a sazonalidade pode ter mudado | Mudanças dentro de cada estação |
| Scatter plot | Quando há covariáveis | Relação entre demanda e variável externa |
| Gráfico de lag | Antes de modelar | Quais defasagens têm correlação |
| ACF | Diagnóstico padrão | Confirma sazonalidade, tendência, período |
| ACF dos resíduos | Após modelar | Verifica se o modelo capturou tudo |

---

## Aplicação direta na sua pesquisa

1. **Time plot de cada série** na hierarquia revela se o padrão é consistente entre níveis
2. **Gráfico sazonal** confirma se produtos têm sazonalidade anual, mensal ou semanal
3. **Matriz de dispersão** entre séries do mesmo nível revela séries correlacionadas
4. **ACF** define o período sazonal correto antes de configurar qualquer modelo
5. **ACF dos resíduos** é o critério de qualidade de qualquer modelo hierárquico

---

## Próximo: Capítulo 3 — Decomposição de Séries Temporais

STL, X11 e SEATS — métodos formais para separar tendência, sazonalidade e resíduo.
É o pré-processamento padrão antes de qualquer modelagem hierárquica.

---
*Notebook gerado como material de estudo para pesquisa de mestrado — Previsão de Demanda com Séries Hierárquicas | UPE*